In [ ]:
# -----------------------------------------------------------------
# "LYTOLLIS 2.0" - THE GROKKING EXPERIMENT (v3)
#
# This version implements the *CORRECT* O(N) URT Optimizer.
#
# Instead of a fixed learning rate, we *calculate* the
# learning rate (Psi strength) at each step based on
# the L2 decay (H strength) and the margin (delta).
#
# lr = (1 - delta) * (alpha * norm(w) / norm(grad))
# -----------------------------------------------------------------

print("--- Step 1: Installing required libraries... ---")
!pip install numpy scikit-learn torch matplotlib &> /dev/null
print("Done.")

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 2: Setup the "Grokking" Task (Modular Arithmetic) ---
print("--- Step 2: Setting up Modular Arithmetic Task ---")

# The prime number for our task
P = 97

# Create all possible pairs (a, b)
a = torch.arange(P)
b = torch.arange(P)
x = torch.cartesian_prod(a, b).float()
# The label is (a + b) % P
y = (x[:, 0] + x[:, 1]) % P
y = y.unsqueeze(1)

# Create a small, held-out test set (20%)
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
test_dataset = torch.utils.data.TensorDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1024, shuffle=False)

print(f"Data created: {len(X_train)} train samples, {len(X_test)} test samples.")
print(f"Task: (a + b) % {P}")


# --- Step 3: Define the Model ---
print("--- Step 3: Defining Neural Network Model ---")

class ModuloNet(nn.Module):
    """A simple MLP with chaotic-friendly 'sin' activation"""
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.lin1 = nn.Linear(2, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, 1)
        self.activation = torch.sin

    def forward(self, x):
        x_norm = (x / P) * (2 * np.pi) - np.pi
        x = self.activation(self.lin1(x_norm))
        x = self.lin2(x)
        return x

loss_fn = nn.MSELoss()

# --- Step 4: The *Correct* Lytollis O(N) URT Optimizer ---
print("--- Step 4: Defining the *Correct* Lytollis URT Optimizer ---")

def lytollis_optimizer_step(model, delta, alpha):
    """
    Performs one optimization step using the Lytollis Law
    by dynamically calculating the learning rate (Psi strength)
    based on the weight decay (H strength).
    """
    with torch.no_grad():

        # O(N) pass to get norms of weights and gradients
        global_weight_norm = 0.0
        global_grad_norm = 0.0

        for param in model.parameters():
            if param.grad is None:
                continue
            global_weight_norm += torch.norm(param.data)**2
            global_grad_norm += torch.norm(param.grad.data)**2

        global_weight_norm = torch.sqrt(global_weight_norm)
        global_grad_norm = torch.sqrt(global_grad_norm)

        # --- The O(N) URT Calculator ---
        # 1. H_strength = Mag(H) = alpha * norm(w)
        H_strength = alpha * global_weight_norm

        # 2. Psi_strength_target = (1 - delta) * H_strength
        Psi_strength_target = (1.0 - delta) * H_strength

        # 3. lr = Psi_strength_target / Mag(grad)
        #    This is the learning rate that *enforces* the law.
        #    Add 1e-8 to prevent division by zero.
        dynamic_lr = Psi_strength_target / (global_grad_norm + 1e-8)
        # --------------------------------

        # Apply the update
        for param in model.parameters():
            if param.grad is None:
                continue

            # H force = L2 decay
            H_force = -alpha * param.data

            # Psi force = scaled gradient step
            Psi_force = -dynamic_lr * param.grad

            # w_t+1 = w_t + H_force + Psi_force
            param.data = param.data + H_force + Psi_force


# --- Step 5: The Training Function ---
print("--- Step 5: Defining Training Loop ---")

def train_model(delta, epochs=15000, alpha=1e-4):
    """
    Trains a model for a given delta.
    The learning rate is now *calculated* by the optimizer.
    """
    print(f"\n=== TRAINING WITH DELTA = {delta} ===")

    model = ModuloNet()

    train_losses = []
    test_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0
        for X_batch, y_batch in train_loader:
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            model.zero_grad()
            loss.backward()

            # --- Lytollis Optimizer Step ---
            # Note: 'lr' is now calculated *inside* this function
            lytollis_optimizer_step(model, delta=delta, alpha=alpha)
            # -------------------------------

            epoch_train_loss += loss.item()

        # --- Evaluate on Test Set ---
        model.eval()
        epoch_test_loss = 0
        with torch.no_grad():
            for X_batch_test, y_batch_test in test_loader:
                y_pred_test = model(X_batch_test)
                # We care about the *real* answer, so we check the mod-accuracy
                # Loss < 100 means it's finding the pattern.
                # Loss > 800 means it's just guessing the average (48).
                # Loss ~3000 means it's guessing 0.
                test_loss = loss_fn(y_pred_test, y_batch_test)
                epoch_test_loss += test_loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_test_loss = epoch_test_loss / len(test_loader)

        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)

        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1:5d} | Train Loss: {avg_train_loss:10.2f} | Test Loss: {avg_test_loss:10.2f}")
            # Grokking check:
            if avg_test_loss < 100 and avg_train_loss > 1000: # High train loss, low test loss
                print(f"    *** GROKKING DETECTED at epoch {epoch+1} ***")

        # Add an early stop for the high-delta case if it converges
        if delta > 0.5 and avg_train_loss < 800: # Converged to the "dumb" mean
             print("High-delta model converged to simple solution. Stopping early.")
             break

    print(f"Final Test Loss for delta={delta}: {avg_test_loss:.2f}")
    return train_losses, test_losses


# --- Step 6: Run the Experiments ---
print("\n--- Step 6: Running Experiments (This will take several minutes) ---")
# Use a strong L2 decay (alpha) to make the H-force significant
ALPHA = 1e-3
EPOCHS = 15000

# Experiment 1: Low-delta (High Exploration)
# We predict this will overfit and then "grok"
train_loss_low, test_loss_low = train_model(
    delta = 0.1,   # "Leash" is long: Mag(Psi) is 90% of Mag(H)
    epochs = EPOCHS,
    alpha = ALPHA
)

# Experiment 2: High-delta (Low Exploration)
# We predict this will converge smoothly to a bad solution
train_loss_high, test_loss_high = train_model(
    delta = 0.9,   # "Leash" is short: Mag(Psi) is 10% of Mag(H)
    epochs = EPOCHS,
    alpha = ALPHA
)

# --- Step 7: Plot the Final Results ---
print("\n--- Step 7: Plotting Final Results ---")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("Lytollis 2.0 (v3): 'Grokking' vs 'Tame' Convergence", fontsize=16)

# --- Low-delta Plot ---
ax1.set_title(f"Low-delta ($\delta=0.1$): 'Grokking' (Predicted)")
ax1.plot(train_loss_low, label='Train Loss', color='blue', alpha=0.8)
ax1.plot(test_loss_low, label='Test Loss', color='orange', linewidth=2)
ax1.set_yscale('log')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.legend()
ax1.grid(True, which="both", ls="--", alpha=0.3)
ax1.set_ylim(bottom=1, top=10000) # Set Y-axis limits

# --- High-delta Plot ---
ax2.set_title(f"High-delta ($\delta=0.9$): 'Tame' Convergence (Predicted)")
ax2.plot(train_loss_high, label='Train Loss', color='blue', alpha=0.8)
ax2.plot(test_loss_high, label='Test Loss', color='orange', linewidth=2)
ax2.set_yscale('log')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- Final Verdict ---
print("\n" + "="*50)
print("             FINAL VERIFICATION (Lytollis 2.0 v3)")
print("="*50)
print(f"Low-delta Final Test Loss:  {test_loss_low[-1]:.2f}")
print(f"High-delta Final Test Loss: {test_loss_high[-1]:.2f}")
print("\n---")
print("  Prediction: Low-delta test loss should show a 'grokking'")
print("  (a U-shape) and end much lower than the high-delta loss.")
print("---")

# A test loss < 100 means it found the pattern.
# A test loss > 800 means it failed and found the "dumb" mean.
if test_loss_low[-1] < 100 and test_loss_high[-1] > 800:
    print("\n  VERDICT: PASSED")
    print("  The data supports the Lytollis 2.0 hypothesis.")
    print("  The Low-delta (chaotic) model successfully 'grokked'")
    print("  the pattern, while the High-delgʻta (tame) model failed.")
else:
    print("\n  VERDICT: FAILED / INCONCLUSIVE")
    print("  The data does not support the hypothesis.")

print("="*50)

<>:224: SyntaxWarning: invalid escape sequence '\d'
<>:235: SyntaxWarning: invalid escape sequence '\d'
<>:224: SyntaxWarning: invalid escape sequence '\d'
<>:235: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-2623996848.py:224: SyntaxWarning: invalid escape sequence '\d'
  ax1.set_title(f"Low-delta ($\delta=0.1$): 'Grokking' (Predicted)")
/tmp/ipython-input-2623996848.py:235: SyntaxWarning: invalid escape sequence '\d'
  ax2.set_title(f"High-delta ($\delta=0.9$): 'Tame' Convergence (Predicted)")


--- Step 1: Installing required libraries... ---
Done.
--- Step 2: Setting up Modular Arithmetic Task ---
Data created: 7527 train samples, 1882 test samples.
Task: (a + b) % 97
--- Step 3: Defining Neural Network Model ---
--- Step 4: Defining the *Correct* Lytollis URT Optimizer ---
--- Step 5: Defining Training Loop ---

--- Step 6: Running Experiments (This will take several minutes) ---

=== TRAINING WITH DELTA = 0.1 ===
Epoch   500 | Train Loss:    2812.89 | Test Loss:    2793.27
Epoch  1000 | Train Loss:    2995.11 | Test Loss:    2974.63
Epoch  1500 | Train Loss:    3048.02 | Test Loss:    3024.78
Epoch  2000 | Train Loss:    3066.61 | Test Loss:    3048.13
Epoch  2500 | Train Loss:    3076.98 | Test Loss:    3059.21
Epoch  3000 | Train Loss:    3087.15 | Test Loss:    3064.45
Epoch  3500 | Train Loss:    3090.76 | Test Loss:    3066.93
Epoch  4000 | Train Loss:    3088.42 | Test Loss:    3068.10
Epoch  4500 | Train Loss:    3087.31 | Test Loss:    3068.66
Epoch  5000 | Train L